# Following one request through DNS, TLS, and a load balancer
> L2 concept exercise — I wanted to see what actually happens when my browser asks a service for data, so I traced a single request through each network layer: DNS resolution (name → IP), the TLS handshake (proving identity + agreeing on encryption), and load-balanced routing (which backend answers). I used only Python's standard library and simulated the backends locally, so nothing here needs a live cluster or a cloud account.

## Setup
I need `socket`, `ssl`, and `http.server` from the standard library, plus `matplotlib` for the drawings (the cells below fall back to ASCII art if it is not installed). Running the next cell checks my environment.

In [ ]:
# last_verified: 2026-08-10 · networking concepts n/a
import socket, ssl
from collections import Counter

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAS_MPL = True
    print(f"matplotlib {matplotlib.__version__} — visualizations enabled")
except ImportError:
    HAS_MPL = False
    print("matplotlib not found — falling back to ASCII diagrams")

## DNS resolution — from name to address
Every journey starts with a name. When I request `example.com`, my machine asks a DNS resolver, which walks the hierarchy (root servers → TLD `.com` servers → the domain's authoritative nameservers) to return one or more IP addresses. I wrote a cell that resolves a hostname and prints every record my resolver hands back — in the order it prefers.

In [ ]:
# last_verified: 2026-08-10 · networking concepts n/a
def resolve_hostname(hostname, port=443):
    """Return all address records the resolver gives us, in order."""
    results = socket.getaddrinfo(hostname, port, socket.AF_UNSPEC, socket.SOCK_STREAM)
    seen = set()
    records = []
    for family, type_, proto, canonname, sockaddr in results:
        ip = sockaddr[0]
        if ip not in seen:
            seen.add(ip)
            family_name = "IPv6" if family == socket.AF_INET6 else "IPv4"
            records.append({"family": family_name, "ip": ip, "port": sockaddr[1]})
    return records

hostname = "example.com"
records = resolve_hostname(hostname)

print(f"DNS resolved {hostname}:")
for i, r in enumerate(records, 1):
    print(f"  {i}. {r['family']}: {r['ip']}:{r['port']}")

In [ ]:
# last_verified: 2026-08-10 · networking concepts n/a
# Visualise the DNS resolution path as a layered flow
if HAS_MPL:
    layers = ["Browser\n" + hostname,
              "DNS Resolver",
              "Root\nServer",
              "TLD\n(.com) Server",
              "Authoritative\nNameserver",
              "IP Address\n" + records[0]['ip'] + ""]
    colors = ["#4CAF50", "#2196F3", "#FF9800", "#FF9800", "#FF9800", "#4CAF50"]
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.bar(range(len(layers)), [1] * len(layers), color=colors)
    for i, label in enumerate(layers):
        ax.text(i, 1.05, label, ha='center', va='bottom', fontsize=8)
    ax.set_xticks(range(len(layers)))
    ax.set_xticklabels(layers, fontsize=7)
    ax.set_yticks([])
    ax.set_title("DNS Resolution Flow — name to IP address")
    plt.tight_layout()
    plt.show()
else:
    print("DNS flow (ASCII fallback):")
    print(f"  Browser ({hostname}) -> Resolver -> Root -> TLD -> Nameserver -> {records[0]['ip']}")

### What I saw
- `getaddrinfo` returned records in the order my resolver prefers — on dual-stack hosts that is usually IPv6 first, then IPv4. The order matters: many libraries try the first address and only fall back if it fails.
- A hostname can map to **several** IPs (round-robin DNS). That is the simplest form of load distribution — each new connection picks the next address in the list.

## TLS handshake — proving who is on the other end
Once I have an IP, I open a TCP connection to port 443. Then TLS kicks in: the server presents its certificate, my client checks that it chains to a trusted root CA and covers the hostname, and the two sides agree on encryption. I used `ssl.create_default_context()` to perform a real handshake against `example.com` and pull the certificate details.

In [ ]:
# last_verified: 2026-08-10 · networking concepts n/a
def tls_handshake(hostname, port=443):
    """Open a TLS connection and return certificate + handshake info."""
    context = ssl.create_default_context()
    with socket.create_connection((hostname, port), timeout=10) as raw_sock:
        with context.wrap_socket(raw_sock, server_hostname=hostname) as tls_sock:
            cert = tls_sock.getpeercert()
            cipher = tls_sock.cipher()
            version = tls_sock.version()
            return {
                "subject": cert.get("subject", []),
                "issuer": cert.get("issuer", []),
                "not_after": cert.get("notAfter", ""),
                "san": cert.get("subjectAltName", []),
                "cipher": cipher,
                "version": version,
            }

info = tls_handshake("example.com")
print(f"TLS version: {info['version']}")
print(f"Cipher: {info['cipher'][0]} / {info['cipher'][1]} / {info['cipher'][2]}")
print(f"Not after: {info['not_after']}")
print(f"SAN entries: {info['san']}")

In [ ]:
# last_verified: 2026-08-10 · networking concepts n/a
# Visualise the TLS handshake steps in order
steps = [
    "Step 1: Client -> Server  TCP SYN",
    "Step 2: Server -> Client  TCP SYN-ACK",
    "Step 3: Client -> Server  TCP ACK  (connection open)",
    "Step 4: Client -> Server  ClientHello (supported TLS versions, ciphers)",
    "Step 5: Server -> Client  ServerHello + Certificate + ServerKeyExchange + ServerHelloDone",
    "Step 6: Client  validates cert, sends ClientKeyExchange + ChangeCipherSpec + Finished",
    "Step 7: Server  ChangeCipherSpec + Finished",
    "Step 8: Encrypted application data flows bi-directionally",
]
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.barh(range(len(steps)), [1] * len(steps), color="#9C27B0")
    ax.set_yticks(range(len(steps)))
    ax.set_yticklabels(steps, fontsize=8)
    ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_title("TLS Handshake — 8 messages before app data flows")
    plt.tight_layout()
    plt.show()
else:
    for i, s in enumerate(steps, 1):
        print(f"  {s}")

### What I saw
- The certificate's `notAfter` field tells me when trust expires. If the date has passed, or the cert is revoked, the handshake aborts before any HTTP byte is sent.
- The SAN (Subject Alternative Name) list tells me every hostname this cert covers. A mismatch between the hostname I requested and the SAN entries is a common failure.
- The cipher-suite string (key exchange / auth / bulk cipher / MAC) tells me how strong the encryption is — older suites get rejected as insecure.

## Load-balanced routing — which backend answers?
Now the request reaches the application layer. A load balancer fronts several backend instances and picks one for each incoming request. I built a tiny round-robin simulator in Python and sent a batch of requests through it to see how traffic spreads across backends.

In [ ]:
# last_verified: 2026-08-10 · networking concepts n/a
class RoundRobinBalancer:
    """Simplest load balancer — cycle through backends in order."""
    def __init__(self, backends):
        self.backends = backends
        self._next = 0
        self.assignments = []

    def next_backend(self):
        backend = self.backends[self._next % len(self.backends)]
        self._next += 1
        self.assignments.append(backend)
        return backend

backends = ["backend-1:8080", "backend-2:8080", "backend-3:8080"]
lb = RoundRobinBalancer(backends)

print("Routing 9 simulated requests:")
for i in range(1, 10):
    backend = lb.next_backend()
    print(f"  Request {i:>2} -> {backend}")

In [ ]:
# last_verified: 2026-08-10 · networking concepts n/a
counts = Counter(lb.assignments)
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(7, 4))
    names = list(counts.keys())
    values = list(counts.values())
    bars = ax.bar(names, values, color="#009688")
    ax.set_ylabel("Request count")
    ax.set_title("Round-robin distribution across 3 backends (9 requests)")
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.1, str(val),
                ha='center', va='bottom')
    plt.tight_layout()
    plt.show()
else:
    print("Distribution (ASCII):")
    for b in backends:
        bar = "#" * counts[b]
        print(f"  {b}: {bar} ({counts[b]})")

### What I saw
- Round-robin spread evenly: 3 requests to each backend. That's the expected behaviour when all backends are equally weighted and healthy.
- The first request always went to `backend-1` — round-robin is stateful (it remembers its position). In a real load balancer that starts cold, the first connection lands on the first backend in the pool.
- Real load balancers do more than round-robin — they track response latency, weight backends differently, and pull unhealthy instances out of the rotation. A single slow or failing backend would skew this neat even spread.

## One request, three layers
| Layer | What happens | How I saw it |
|---|---|---|
| DNS | Name resolves to one or more IPs | `socket.getaddrinfo` returns address records |
| TLS | Certificate is validated, encryption agreed | `ssl.create_default_context().wrap_socket` returns cert details |
| Load balancer | Backend selected from a pool | Round-robin counter cycles through backends |

Each layer adds its own latency and its own failure mode. If a request fails, the question is always: which layer dropped it? Was DNS wrong? Did the cert expire? Did the chosen backend crash?

## What I'd try next
- Replace the round-robin simulator with an actual reverse proxy (Python `http.server` or nginx) and watch requests land on different backends.
- Use the `ssl` module's `getpeercert(binary_form=True)` with the `cryptography` library to parse the full certificate chain and check for expiration or revocation.
- Capture real packets with `tcpdump` or trace the route with `traceroute` to see the actual network path, not just the Python-level simulation.